# 00.5 Conservative Pair Feasibility Gate

Upload one original dog image and one replacement-dog image. This notebook independently runs the key logic needed for pair screening:

- replacement-oriented pose analysis
- segmentation and cutout pose re-check
- pose-guided fitting diagnostics
- visual explanation of how much the replacement pose must bend to match the target pose


In [ ]:
%matplotlib inline


## What This Module Produces

- direct upload-driven pair analysis
- global pose summary for the original and replacement dog
- segmentation and cutout previews inspired by notebook `01`
- cutout-pose re-estimation on cleaner foreground-only crops
- fitting diagnostics inspired by notebook `02`
- a presentation-friendly overlay showing:
  - target pose
  - similarity-seed pose
  - final fitted pose
- a table with readable scores and visual bars


In [ ]:
import importlib.util
import json
import math
import os
from datetime import datetime
from pathlib import Path

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import torch
from IPython.display import HTML, display, clear_output
from PIL import Image
from skimage.transform import PiecewiseAffineTransform, SimilarityTransform
from ultralytics import SAM, YOLO


def has_module(module_name):
    return importlib.util.find_spec(module_name) is not None


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the current notebook path.")


NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = find_repo_root(NOTEBOOK_DIR)
OUTPUT_ROOT = ROOT / "data" / "outputs" / "00_5_pair_feasibility"
UPLOAD_ROOT = OUTPUT_ROOT / "_uploaded_pairs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

POSE_MODEL_PATH = Path(os.environ.get("DOG_POSE_MODEL_PATH", str(ROOT / "models" / "dog_pose_best.pt"))).resolve()


def resolve_weight(project_root, filename):
    candidates = [
        project_root / "models" / filename,
        project_root / "notebooks" / filename,
        project_root / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return Path(filename)


YOLO_DET_WEIGHTS = resolve_weight(ROOT, "yolov8n.pt")
YOLO_SEG_WEIGHTS = resolve_weight(ROOT, "yolov8x-seg.pt")
SAM_WEIGHTS = resolve_weight(ROOT, "sam_b.pt")

POSE_CONF = 0.15
POSE_IMGSZ = 960
KEYPOINT_CONF = 0.20

DEVICE = 0 if torch.cuda.is_available() else None
if DEVICE is None:
    raise RuntimeError("This notebook is local-GPU only. CUDA was not detected.")
if not POSE_MODEL_PATH.exists():
    raise FileNotFoundError(f"Dog pose model not found: {POSE_MODEL_PATH}")

print("Project root:", ROOT)
print("Output root:", OUTPUT_ROOT)
print("YOLO detector:", YOLO_DET_WEIGHTS)
print("YOLO segmenter:", YOLO_SEG_WEIGHTS)
print("SAM weights:", SAM_WEIGHTS)
print("Pose model:", POSE_MODEL_PATH)
print("CUDA device:", torch.cuda.get_device_name(0))


## Shared Pose Definitions


In [ ]:
DOG_KEYPOINT_NAMES = [
    "front_left_paw",
    "front_left_knee",
    "front_left_elbow",
    "rear_left_paw",
    "rear_left_knee",
    "rear_left_elbow",
    "front_right_paw",
    "front_right_knee",
    "front_right_elbow",
    "rear_right_paw",
    "rear_right_knee",
    "rear_right_elbow",
    "tail_start",
    "tail_end",
    "left_ear_base",
    "right_ear_base",
    "nose",
    "chin",
    "left_ear_tip",
    "right_ear_tip",
    "left_eye",
    "right_eye",
    "withers",
    "throat",
]
NAME_TO_INDEX = {name: idx for idx, name in enumerate(DOG_KEYPOINT_NAMES)}

PART_GROUPS = {
    "head": ["nose", "chin", "left_eye", "right_eye", "left_ear_base", "right_ear_base", "left_ear_tip", "right_ear_tip"],
    "torso": ["withers", "throat", "tail_start"],
    "front_legs": ["front_left_paw", "front_left_knee", "front_left_elbow", "front_right_paw", "front_right_knee", "front_right_elbow"],
    "hind_legs": ["rear_left_paw", "rear_left_knee", "rear_left_elbow", "rear_right_paw", "rear_right_knee", "rear_right_elbow"],
    "tail": ["tail_start", "tail_end"],
}

POSE_EDGES = [
    ("nose", "chin"),
    ("nose", "left_eye"),
    ("nose", "right_eye"),
    ("left_eye", "left_ear_base"),
    ("right_eye", "right_ear_base"),
    ("left_ear_base", "left_ear_tip"),
    ("right_ear_base", "right_ear_tip"),
    ("throat", "nose"),
    ("throat", "withers"),
    ("withers", "tail_start"),
    ("tail_start", "tail_end"),
    ("withers", "front_left_elbow"),
    ("front_left_elbow", "front_left_knee"),
    ("front_left_knee", "front_left_paw"),
    ("withers", "front_right_elbow"),
    ("front_right_elbow", "front_right_knee"),
    ("front_right_knee", "front_right_paw"),
    ("tail_start", "rear_left_elbow"),
    ("rear_left_elbow", "rear_left_knee"),
    ("rear_left_knee", "rear_left_paw"),
    ("tail_start", "rear_right_elbow"),
    ("rear_right_elbow", "rear_right_knee"),
    ("rear_right_knee", "rear_right_paw"),
]


## Core Helpers

This block merges the necessary pieces from notebooks `00`, `01`, and `02` into one self-contained pair-analysis workflow.


In [ ]:
def read_rgb(path: Path) -> np.ndarray:
    arr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if arr is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    return cv2.cvtColor(arr, cv2.COLOR_BGR2RGB)


def ensure_uint8(image):
    return np.clip(image, 0, 255).astype(np.uint8)


def save_rgb(path, image):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(ensure_uint8(image)).save(path)


def save_mask(path, mask):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray((mask.astype(np.uint8) * 255)).save(path)


def is_valid_point(point):
    return (
        point is not None
        and len(point) == 2
        and point[0] is not None
        and point[1] is not None
        and not (np.isnan(point[0]) or np.isnan(point[1]))
    )


def clip_bbox(xyxy, width, height):
    x1, y1, x2, y2 = [float(v) for v in xyxy]
    x1 = max(0.0, min(x1, width - 1))
    y1 = max(0.0, min(y1, height - 1))
    x2 = max(x1 + 1.0, min(x2, width))
    y2 = max(y1 + 1.0, min(y2, height))
    return [x1, y1, x2, y2]


def bbox_area(bbox):
    x1, y1, x2, y2 = bbox
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def bbox_width_height(bbox):
    x1, y1, x2, y2 = bbox
    return max(1e-6, x2 - x1), max(1e-6, y2 - y1)


def bbox_diag(bbox):
    w, h = bbox_width_height(bbox)
    return float(np.linalg.norm([w, h]))


def bbox_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    inter = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    return inter / max(1, bbox_area(box_a) + bbox_area(box_b) - inter)


def mask_to_bbox(mask):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None
    return [int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1]


def draw_bbox(image, bbox, color=(255, 80, 80), thickness=4):
    out = image.copy()
    x1, y1, x2, y2 = [int(v) for v in bbox]
    cv2.rectangle(out, (x1, y1), (x2, y2), color, thickness)
    return out


def overlay_mask(image, mask, color=(30, 144, 255), alpha=0.45):
    out = image.astype(np.float32).copy()
    color_arr = np.array(color, dtype=np.float32)
    out[mask] = out[mask] * (1.0 - alpha) + color_arr * alpha
    return ensure_uint8(out)


def rgba_on_checkerboard(rgba):
    h, w = rgba.shape[:2]
    tile = 16
    yy, xx = np.indices((h, w))
    board = ((yy // tile + xx // tile) % 2).astype(np.uint8)
    checker = np.where(board[..., None] == 0, 220, 180).astype(np.uint8)
    checker = np.repeat(checker[..., :1], 3, axis=2)
    alpha = rgba[..., 3:4].astype(np.float32) / 255.0
    return ensure_uint8(rgba[..., :3].astype(np.float32) * alpha + checker.astype(np.float32) * (1.0 - alpha))


def point_from_arrays(points, confs, name, threshold=KEYPOINT_CONF):
    idx = NAME_TO_INDEX[name]
    if idx >= len(points):
        return None
    point = points[idx]
    if point is None:
        return None
    point = np.array(point, dtype=np.float32)
    if point.shape != (2,) or np.isnan(point).any():
        return None
    conf = float(confs[idx]) if confs is not None and idx < len(confs) else 1.0
    if conf < threshold:
        return None
    return point


def orientation_label(points, confs):
    nose = point_from_arrays(points, confs, "nose")
    tail_start = point_from_arrays(points, confs, "tail_start")
    if nose is None or tail_start is None:
        return "unknown"
    return "left" if nose[0] < tail_start[0] else "right"


def frame_fit_bucket(bbox, width, height, margin=4):
    x1, y1, x2, y2 = bbox
    touch_count = int(x1 <= margin) + int(y1 <= margin) + int(x2 >= width - margin) + int(y2 >= height - margin)
    if touch_count > 0:
        return "truncated_by_frame", touch_count
    return "in_frame", touch_count


def visible_part_signature(points, confs):
    visible_parts = []
    rules = {
        "head": 2,
        "torso": 1,
        "front_legs": 2,
        "hind_legs": 2,
        "tail": 1,
    }
    for part_name, point_names in PART_GROUPS.items():
        visible_count = sum(point_from_arrays(points, confs, name) is not None for name in point_names)
        if visible_count >= rules[part_name]:
            visible_parts.append(part_name)
    return "+".join(visible_parts) if visible_parts else "none"


def coverage_class(points, confs):
    signature = visible_part_signature(points, confs)
    required = {"head", "torso", "front_legs", "hind_legs", "tail"}
    present = set(signature.split("+")) if signature != "none" else set()
    return "full_body" if required.issubset(present) else "partial_body"


def body_axis(points, confs):
    nose = point_from_arrays(points, confs, "nose")
    tail_start = point_from_arrays(points, confs, "tail_start")
    if nose is None or tail_start is None:
        return None
    return nose - tail_start


def body_angle_deg(points, confs):
    axis = body_axis(points, confs)
    if axis is None:
        return None
    return float(np.degrees(np.arctan2(axis[1], axis[0])))


def body_angle_bucket(points, confs):
    angle = body_angle_deg(points, confs)
    if angle is None:
        return "unknown"
    abs_angle = abs(angle) % 180.0
    if abs_angle > 90.0:
        abs_angle = 180.0 - abs_angle
    if abs_angle <= 20:
        return "horizontal"
    if abs_angle <= 55:
        return "diagonal"
    return "vertical"


def posture_bucket(bbox):
    width, height = bbox_width_height(bbox)
    ratio = height / max(width, 1e-6)
    if ratio < 0.62:
        return "low"
    if ratio > 0.92:
        return "tall"
    return "mid"


def spread_bucket(points, confs, bbox):
    paws = [
        point_from_arrays(points, confs, "front_left_paw"),
        point_from_arrays(points, confs, "front_right_paw"),
        point_from_arrays(points, confs, "rear_left_paw"),
        point_from_arrays(points, confs, "rear_right_paw"),
    ]
    paws = [pt for pt in paws if pt is not None]
    width, _ = bbox_width_height(bbox)
    if len(paws) < 2:
        return "unknown"
    xs = np.array([pt[0] for pt in paws], dtype=np.float32)
    normalized_span = float((xs.max() - xs.min()) / max(width, 1e-6))
    if normalized_span >= 0.62:
        return "extended"
    if normalized_span >= 0.42:
        return "balanced"
    return "compact"


def classify_view(points, confs, bbox):
    orientation = orientation_label(points, confs)
    signature = visible_part_signature(points, confs)
    coverage = coverage_class(points, confs)
    angle_bucket = body_angle_bucket(points, confs)
    width, height = bbox_width_height(bbox)
    bbox_ratio = width / max(height, 1e-6)
    if orientation != "unknown" and coverage == "full_body" and "front_legs" in signature and "hind_legs" in signature and bbox_ratio > 1.0:
        return "side_profile"
    if orientation == "unknown" and angle_bucket == "vertical":
        return "frontal_or_rear"
    return "ambiguous"


def canonical_normalized_keypoints(points, confs, bbox):
    valid_points = [point_from_arrays(points, confs, name, threshold=-1.0) for name in DOG_KEYPOINT_NAMES]
    valid_points = [pt for pt in valid_points if pt is not None]
    if not valid_points:
        return [[None, None] for _ in DOG_KEYPOINT_NAMES]

    tail_start = point_from_arrays(points, confs, "tail_start")
    withers = point_from_arrays(points, confs, "withers")
    nose = point_from_arrays(points, confs, "nose")

    if tail_start is not None and withers is not None:
        center = 0.5 * (tail_start + withers)
    else:
        center = np.mean(np.stack(valid_points), axis=0)

    if tail_start is not None and nose is not None:
        scale = float(np.linalg.norm(nose - tail_start))
    else:
        box_w, box_h = bbox_width_height(bbox)
        scale = max(box_w, box_h)
    scale = max(scale, 1e-6)

    orientation = orientation_label(points, confs)
    normalized = []
    for name in DOG_KEYPOINT_NAMES:
        pt = point_from_arrays(points, confs, name, threshold=-1.0)
        if pt is None:
            normalized.append([None, None])
            continue
        value = (pt - center) / scale
        if orientation == "right":
            value = np.array([-value[0], value[1]], dtype=np.float32)
        normalized.append([float(value[0]), float(value[1])])
    return normalized


def record_point(record, name, field="keypoints_xy"):
    idx = NAME_TO_INDEX[name]
    points = record.get(field) or []
    if idx >= len(points):
        return None
    point = points[idx]
    if not is_valid_point(point):
        return None
    return np.array(point, dtype=np.float32)


def shape_vector_from_normalized(record):
    def norm_point(name):
        return record_point(record, name, field="normalized_keypoints")

    xs = []
    ys = []
    for point in record.get("normalized_keypoints") or []:
        if is_valid_point(point):
            xs.append(point[0])
            ys.append(point[1])
    if not xs:
        return [None] * 8

    width = float(max(xs) - min(xs))
    height = float(max(ys) - min(ys))
    aspect = width / max(height, 1e-6)

    def segment_length(a_name, b_name):
        a = norm_point(a_name)
        b = norm_point(b_name)
        if a is None or b is None:
            return None
        return float(np.linalg.norm(a - b))

    def mean_optional(values):
        values = [float(v) for v in values if v is not None and not np.isnan(v)]
        if not values:
            return None
        return float(np.mean(values))

    body_len = segment_length("nose", "tail_start")
    front_leg = mean_optional([
        segment_length("front_left_elbow", "front_left_paw"),
        segment_length("front_right_elbow", "front_right_paw"),
    ])
    hind_leg = mean_optional([
        segment_length("rear_left_elbow", "rear_left_paw"),
        segment_length("rear_right_elbow", "rear_right_paw"),
    ])
    head_len = mean_optional([
        segment_length("nose", "chin"),
        segment_length("nose", "left_ear_base"),
        segment_length("nose", "right_ear_base"),
    ])
    tail_len = segment_length("tail_start", "tail_end")

    return [
        width,
        height,
        aspect,
        float(body_len) if body_len is not None else None,
        float(front_leg) if front_leg is not None else None,
        float(hind_leg) if hind_leg is not None else None,
        float(head_len) if head_len is not None else None,
        float(tail_len) if tail_len is not None else None,
    ]


def shape_label_from_vector(vector):
    aspect = vector[2]
    front_leg = vector[4]
    hind_leg = vector[5]
    if aspect is None:
        return "unknown"
    leg_values = [float(v) for v in [front_leg, hind_leg] if v is not None and not np.isnan(v)]
    mean_leg = float(np.mean(leg_values)) if leg_values else None
    if aspect >= 2.15 and (mean_leg is None or mean_leg >= 0.55):
        return "slender"
    if aspect <= 1.35:
        return "compact"
    if mean_leg is not None and mean_leg >= 0.72:
        return "long_legged"
    return "balanced"


def current_stage_gate(record):
    signature = record["visible_parts_signature"]
    required_parts = {"head", "torso", "front_legs", "hind_legs", "tail"}
    visible_parts = set(signature.split("+")) if signature != "none" else set()
    if record["status"] != "ok":
        return False, "no_pose"
    if record["dog_instance_count"] > 1:
        return False, "multiple_dogs_present"
    if record["view_class"] != "side_profile":
        return False, "not_side_profile"
    if record["coverage_class"] != "full_body":
        return False, "not_full_body"
    if record["frame_fit_bucket"] != "in_frame":
        return False, "truncated_by_frame"
    if record["visible_keypoints"] < 12:
        return False, "too_few_keypoints"
    if not required_parts.issubset(visible_parts):
        return False, "missing_required_parts"
    if record["bbox_area_ratio"] > 0.55:
        return False, "dog_too_large_in_frame"
    return True, "pass"


det_model = YOLO(str(YOLO_DET_WEIGHTS))
seg_model = YOLO(str(YOLO_SEG_WEIGHTS))
sam_model = SAM(str(SAM_WEIGHTS))
pose_model = YOLO(str(POSE_MODEL_PATH))


def get_keypoint_arrays(result, instance_idx):
    xy = result.keypoints.xy[instance_idx].detach().cpu().numpy().astype(np.float32)
    conf_tensor = getattr(result.keypoints, "conf", None)
    if conf_tensor is not None:
        conf = conf_tensor[instance_idx].detach().cpu().numpy().astype(np.float32)
    else:
        data = result.keypoints.data[instance_idx].detach().cpu().numpy().astype(np.float32)
        conf = data[:, 2] if data.shape[-1] >= 3 else np.ones(len(xy), dtype=np.float32)
    return xy, conf


def choose_best_pose_instance(result):
    if result.boxes is None or result.keypoints is None or len(result.boxes) == 0:
        return None
    candidates = []
    for idx, box in enumerate(result.boxes):
        xyxy = box.xyxy[0].detach().cpu().numpy().astype(np.float32)
        area = max(1.0, (xyxy[2] - xyxy[0]) * (xyxy[3] - xyxy[1]))
        score = float(box.conf.item())
        visible = float((get_keypoint_arrays(result, idx)[1] >= KEYPOINT_CONF).sum())
        candidates.append((idx, score * math.sqrt(area) * (1.0 + visible / 24.0)))
    return max(candidates, key=lambda item: item[1])[0]


def query_pose_record(image_path: Path, pose_model):
    image = read_rgb(image_path)
    height, width = image.shape[:2]
    result = pose_model.predict(source=image, conf=POSE_CONF, imgsz=POSE_IMGSZ, device=DEVICE, verbose=False)[0]

    boxes = result.boxes
    if boxes is None or len(boxes) == 0:
        return {
            "status": "no_pose",
            "query_image_path": str(image_path),
            "width": width,
            "height": height,
            "dog_instance_count": 0,
        }

    keypoints_xy_all = result.keypoints.xy.detach().cpu().numpy() if result.keypoints is not None else None
    keypoints_conf_all = result.keypoints.conf.detach().cpu().numpy() if result.keypoints is not None else None

    candidates = []
    for idx, box in enumerate(boxes):
        xyxy = clip_bbox(box.xyxy[0].detach().cpu().numpy().tolist(), width, height)
        score = float(box.conf.item())
        area = bbox_area(xyxy)
        keypoints_xy = keypoints_xy_all[idx].tolist() if keypoints_xy_all is not None else [[None, None] for _ in DOG_KEYPOINT_NAMES]
        keypoints_conf = keypoints_conf_all[idx].tolist() if keypoints_conf_all is not None else [0.0] * len(DOG_KEYPOINT_NAMES)
        candidates.append({
            "bbox_xyxy": xyxy,
            "detection_conf": score,
            "bbox_area": area,
            "keypoints_xy": keypoints_xy,
            "keypoints_conf": keypoints_conf,
        })

    dog_instance_count = len(candidates)
    best = max(candidates, key=lambda item: item["detection_conf"] * math.sqrt(max(item["bbox_area"], 1e-6)))
    points = best["keypoints_xy"]
    confs = best["keypoints_conf"]
    visible_names = [name for name, conf in zip(DOG_KEYPOINT_NAMES, confs) if conf >= KEYPOINT_CONF]
    frame_bucket, edge_touch_count = frame_fit_bucket(best["bbox_xyxy"], width, height)
    geometry_label = "|".join([
        body_angle_bucket(points, confs),
        posture_bucket(best["bbox_xyxy"]),
        spread_bucket(points, confs, best["bbox_xyxy"]),
    ])

    record = {
        "status": "ok",
        "query_image_path": str(image_path),
        "width": width,
        "height": height,
        "dog_instance_count": dog_instance_count,
        "bbox_xyxy": best["bbox_xyxy"],
        "bbox_area_ratio": float(best["bbox_area"] / max(width * height, 1e-6)),
        "detection_conf": best["detection_conf"],
        "keypoints_xy": points,
        "keypoints_conf": confs,
        "visible_keypoints": len(visible_names),
        "visible_keypoint_names": visible_names,
        "visible_parts_signature": visible_part_signature(points, confs),
        "coverage_class": coverage_class(points, confs),
        "original_orientation": orientation_label(points, confs),
        "view_class": classify_view(points, confs, best["bbox_xyxy"]),
        "frame_fit_bucket": frame_bucket,
        "frame_edge_touch_count": edge_touch_count,
        "pose_geometry_label": geometry_label,
        "body_angle_deg": body_angle_deg(points, confs),
        "normalized_keypoints": canonical_normalized_keypoints(points, confs, best["bbox_xyxy"]),
    }
    record["shape_vector"] = shape_vector_from_normalized(record)
    record["shape_label"] = shape_label_from_vector(record["shape_vector"])
    record["current_pipeline_pass"], record["current_pipeline_reason"] = current_stage_gate(record)
    return record


def normalized_array_from_record(record):
    arr = []
    for point in record.get("normalized_keypoints") or []:
        if is_valid_point(point):
            arr.append([float(point[0]), float(point[1])])
        else:
            arr.append([np.nan, np.nan])
    return np.array(arr, dtype=np.float32)


def keypoint_distance(record_a, record_b, min_shared=10):
    a = normalized_array_from_record(record_a)
    b = normalized_array_from_record(record_b)
    shared = np.isfinite(a[:, 0]) & np.isfinite(a[:, 1]) & np.isfinite(b[:, 0]) & np.isfinite(b[:, 1])
    shared_count = int(shared.sum())
    if shared_count < min_shared:
        return math.inf, shared_count
    diff = a[shared] - b[shared]
    dist = float(np.sqrt((diff ** 2).sum(axis=1).mean()))
    return dist, shared_count


def visibility_overlap(record_a, record_b):
    vis_a = set(record_a.get("visible_keypoint_names") or [])
    vis_b = set(record_b.get("visible_keypoint_names") or [])
    if not vis_a and not vis_b:
        return 0.0
    return float(len(vis_a & vis_b)) / max(len(vis_a | vis_b), 1)


def pose_score_bundle(record_a, record_b):
    kp_dist, shared = keypoint_distance(record_a, record_b)
    vis_overlap = visibility_overlap(record_a, record_b)
    if not math.isfinite(kp_dist):
        return {
            "pose_score": None,
            "normalized_keypoint_distance": None,
            "shared_keypoints": int(shared),
            "visibility_overlap": float(vis_overlap),
        }
    score = (0.84 * kp_dist) + (0.16 * (1.0 - vis_overlap))
    return {
        "pose_score": float(score),
        "normalized_keypoint_distance": float(kp_dist),
        "shared_keypoints": int(shared),
        "visibility_overlap": float(vis_overlap),
    }


def evaluate_pair(org_record, rep_record):
    score_bundle = pose_score_bundle(org_record, rep_record)
    warnings = []
    approved = True

    if not org_record.get("current_pipeline_pass"):
        approved = False
        warnings.append(f"original::{org_record.get('current_pipeline_reason', 'not_current_stage')}")
    if not rep_record.get("current_pipeline_pass"):
        approved = False
        warnings.append(f"replacement::{rep_record.get('current_pipeline_reason', 'not_current_stage')}")

    if org_record["visible_parts_signature"] != rep_record["visible_parts_signature"]:
        approved = False
        warnings.append("visible_parts_mismatch")
    if org_record["view_class"] != "side_profile" or rep_record["view_class"] != "side_profile":
        approved = False
        warnings.append("non_side_profile_present")
    if org_record["coverage_class"] != "full_body" or rep_record["coverage_class"] != "full_body":
        approved = False
        warnings.append("non_full_body_present")

    pose_score = score_bundle["pose_score"]
    if pose_score is None:
        approved = False
        warnings.append("too_few_shared_keypoints")
    else:
        if score_bundle["shared_keypoints"] < 12:
            approved = False
            warnings.append("shared_keypoints_below_12")
        if pose_score > 0.105:
            approved = False
            warnings.append("pose_distance_too_large")
        elif pose_score > 0.080:
            warnings.append("pose_distance_borderline")

    scale_ratio = org_record["bbox_area_ratio"] / max(1e-6, rep_record["bbox_area_ratio"])
    if scale_ratio < 0.45 or scale_ratio > 2.30:
        approved = False
        warnings.append("bbox_scale_ratio_out_of_range")
    elif scale_ratio < 0.60 or scale_ratio > 1.80:
        warnings.append("bbox_scale_ratio_borderline")

    recommended_transform_mode = "piecewise_affine"
    if pose_score is None or pose_score > 0.060 or score_bundle["shared_keypoints"] < 15:
        recommended_transform_mode = "similarity_only"
    if not approved:
        recommended_transform_mode = "similarity_only"

    primary_reason = "approved" if approved else (warnings[0] if warnings else "rejected")
    return {
        **score_bundle,
        "approved_for_01": bool(approved),
        "recommended_transform_mode": recommended_transform_mode,
        "primary_reason": primary_reason,
        "warnings": warnings,
        "bbox_scale_ratio": float(scale_ratio),
    }


def plot_images(items, cols=3, figsize=(16, 10)):
    if not items:
        return
    rows = int(math.ceil(len(items) / cols))
    fig = plt.figure(figsize=figsize)
    for idx, (title, image) in enumerate(items, start=1):
        ax = fig.add_subplot(rows, cols, idx)
        if image.ndim == 2:
            ax.imshow(image, cmap="gray")
        else:
            ax.imshow(image)
        ax.set_title(title)
        ax.axis("off")
    fig.tight_layout()
    return fig


def draw_pose_on_axis(ax, pose_record, color="#00C2A8", alpha=1.0, linewidth=2.0, linestyle="-", show_bbox=True):
    if show_bbox:
        x1, y1, x2, y2 = pose_record["bbox_xyxy"]
        ax.add_patch(
            patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                fill=False, edgecolor=color, linewidth=2.0, linestyle=linestyle, alpha=alpha
            )
        )
    point_map = {
        name: pt
        for name, pt, conf in zip(DOG_KEYPOINT_NAMES, pose_record["points"], pose_record["confs"])
        if conf >= KEYPOINT_CONF
    }
    for start_name, end_name in POSE_EDGES:
        start = point_map.get(start_name)
        end = point_map.get(end_name)
        if start is None or end is None:
            continue
        ax.plot([start[0], end[0]], [start[1], end[1]], color=color, linewidth=linewidth, alpha=alpha, linestyle=linestyle)
    for pt in point_map.values():
        ax.scatter(pt[0], pt[1], s=24, color=color, alpha=alpha, edgecolors="white", linewidths=0.35)


def render_dual_pose_figure(org_record, rep_record):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, record, color, panel_title in [
        (axes[0], org_record, "#00C2A8", "Original image pose"),
        (axes[1], rep_record, "#FF7F50", "Replacement image pose"),
    ]:
        image = read_rgb(Path(record["query_image_path"]))
        ax.imshow(image)
        ax.axis("off")
        draw_pose_on_axis(ax, {
            "bbox_xyxy": record["bbox_xyxy"],
            "points": np.array(record["keypoints_xy"], dtype=np.float32),
            "confs": np.array(record["keypoints_conf"], dtype=np.float32),
        }, color=color, alpha=0.95, linewidth=2.2)
        ax.set_title(panel_title)
    fig.tight_layout()
    return fig


def plot_normalized_pose_overlay(org_record, rep_record, figsize=(7, 7)):
    fig, ax = plt.subplots(figsize=figsize)
    colors = [("#1f77b4", org_record, "Original"), ("#d62728", rep_record, "Replacement")]
    for color, record, label in colors:
        point_lookup = {name: record_point(record, name, field="normalized_keypoints") for name in DOG_KEYPOINT_NAMES}
        for a_name, b_name in POSE_EDGES:
            a = point_lookup[a_name]
            b = point_lookup[b_name]
            if a is not None and b is not None:
                ax.plot([a[0], b[0]], [a[1], b[1]], color=color, linewidth=1.5, alpha=0.75)
        xs = []
        ys = []
        for point in point_lookup.values():
            if point is not None:
                xs.append(point[0])
                ys.append(point[1])
        if xs:
            ax.scatter(xs, ys, s=24, color=color, label=label)
    ax.invert_yaxis()
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.25)
    ax.legend()
    ax.set_title("Canonical normalized pose overlay")
    return fig


def extract_upload_entry(upload_value):
    if isinstance(upload_value, dict):
        values = list(upload_value.values())
    else:
        values = list(upload_value)
    if not values:
        return None
    return values[0]


def get_upload_name(entry):
    if entry is None:
        return None
    return entry.get("name") or entry.get("metadata", {}).get("name") or "uploaded_image.png"


def get_upload_bytes(entry):
    if entry is None:
        return None
    content = entry.get("content")
    if isinstance(content, memoryview):
        return content.tobytes()
    if isinstance(content, bytes):
        return content
    if content is None and "value" in entry:
        value = entry["value"]
        if isinstance(value, memoryview):
            return value.tobytes()
        if isinstance(value, bytes):
            return value
    return bytes(content)


def save_upload_entry(entry, dst_dir: Path, prefix: str):
    dst_dir.mkdir(parents=True, exist_ok=True)
    name = get_upload_name(entry)
    raw = get_upload_bytes(entry)
    suffix = Path(name).suffix or ".png"
    safe_name = "".join(ch if ch.isalnum() or ch in ("-", "_", ".") else "_" for ch in Path(name).stem)
    target = dst_dir / f"{prefix}_{safe_name}{suffix}"
    target.write_bytes(raw)
    return target


def score_chip(label, value, color):
    return (
        f"<span style='display:inline-block;padding:4px 10px;border-radius:999px;"
        f"background:{color};color:white;font-size:12px;font-weight:600;margin-right:8px'>{label}: {value}</span>"
    )


def metric_bar_html(label, normalized_value, value_text, status, status_color, note):
    normalized_value = max(0.0, min(1.0, normalized_value))
    fill = int(round(100 * normalized_value))
    return f'''
    <tr>
      <td style="padding:10px 12px;font-weight:600;vertical-align:top">{label}</td>
      <td style="padding:10px 12px;min-width:260px">
        <div style="background:#ECEFF4;border-radius:999px;height:12px;overflow:hidden">
          <div style="width:{fill}%;background:{status_color};height:12px"></div>
        </div>
      </td>
      <td style="padding:10px 12px;white-space:nowrap">{value_text}</td>
      <td style="padding:10px 12px;color:{status_color};font-weight:700;white-space:nowrap">{status}</td>
      <td style="padding:10px 12px">{note}</td>
    </tr>
    '''


def render_html_table(title, rows):
    body = "".join(rows)
    html = f'''
    <div style="margin:12px 0 22px 0">
      <h3 style="margin:0 0 10px 0">{title}</h3>
      <table style="border-collapse:collapse;width:100%;font-size:14px">
        <thead>
          <tr style="background:#F6F8FA;text-align:left">
            <th style="padding:10px 12px">Metric</th>
            <th style="padding:10px 12px">Visual cue</th>
            <th style="padding:10px 12px">Value</th>
            <th style="padding:10px 12px">Assessment</th>
            <th style="padding:10px 12px">Interpretation</th>
          </tr>
        </thead>
        <tbody>{body}</tbody>
      </table>
    </div>
    '''
    return HTML(html)


def assess_keypoint_distance(value):
    if value is None:
        return 0.0, "Poor", "#D1242F", "Too few shared normalized keypoints to trust the pair."
    if value <= 0.060:
        return 0.92, "Good", "#1A7F37", "The canonical poses are tightly aligned."
    if value <= 0.080:
        return 0.72, "Borderline", "#9A6700", "The poses are usable but already noticeably different."
    if value <= 0.105:
        return 0.45, "Risky", "#BC4C00", "The pair may require a conservative transform only."
    return 0.12, "Poor", "#D1242F", "The pose mismatch is too large for the current pipeline."


def assess_visibility_overlap(value):
    if value >= 0.92:
        return 0.95, "Good", "#1A7F37", "The same keypoints are visible in both dogs."
    if value >= 0.82:
        return 0.75, "Borderline", "#9A6700", "Most visible parts overlap, but some structure differs."
    if value >= 0.68:
        return 0.45, "Risky", "#BC4C00", "The pair shares only part of the visible structure."
    return 0.12, "Poor", "#D1242F", "The visibility pattern is too different."


def assess_shared_keypoints(value):
    if value >= 18:
        return 0.95, "Good", "#1A7F37", "Many shared keypoints support reliable pose comparison."
    if value >= 15:
        return 0.75, "Borderline", "#9A6700", "Enough shared keypoints, but not abundant."
    if value >= 12:
        return 0.45, "Risky", "#BC4C00", "The pair barely clears the current minimum."
    return 0.12, "Poor", "#D1242F", "Too few shared keypoints for safe pose-guided alignment."


def assess_scale_ratio(value):
    if 0.75 <= value <= 1.35:
        return 0.95, "Good", "#1A7F37", "The dogs occupy a similar fraction of the frame."
    if 0.60 <= value <= 1.80:
        return 0.72, "Borderline", "#9A6700", "The size gap is workable but not ideal."
    if 0.45 <= value <= 2.30:
        return 0.42, "Risky", "#BC4C00", "The scale gap may force visible stretching."
    return 0.12, "Poor", "#D1242F", "The frame occupancy difference is too large."


def assess_boolean_match(ok, success_note, fail_note):
    if ok:
        return 0.95, "Good", "#1A7F37", success_note
    return 0.15, "Poor", "#D1242F", fail_note


def assess_relative_rmse(value):
    if value <= 0.08:
        return 0.92, "Good", "#1A7F37", "The coarse similarity fit already aligns well."
    if value <= 0.12:
        return 0.70, "Borderline", "#9A6700", "The similarity fit is workable but not tight."
    if value <= 0.24:
        return 0.38, "Risky", "#BC4C00", "The fit error is large and may require a conservative transform."
    return 0.12, "Poor", "#D1242F", "The similarity fit is too weak for reliable pose-guided warping."


def assess_warp_distortion_ratio(value):
    if value <= 0.03:
        return 0.95, "Low distortion", "#1A7F37", "The final fitted pose stays close to the similarity seed."
    if value <= 0.06:
        return 0.72, "Moderate distortion", "#9A6700", "The fitting introduces a noticeable but still controlled deformation."
    if value <= 0.10:
        return 0.42, "High distortion", "#BC4C00", "The fitting is visibly bending the skeleton."
    return 0.12, "Very high distortion", "#D1242F", "The final fit is strongly warping the replacement pose."


def render_pair_metric_table(org_record, rep_record, evaluation, warp_diag=None):
    rows = []
    score_norm, score_status, score_color, score_note = assess_keypoint_distance(evaluation["normalized_keypoint_distance"])
    rows.append(metric_bar_html(
        "Normalized keypoint distance",
        score_norm,
        "n/a" if evaluation["normalized_keypoint_distance"] is None else f"{evaluation['normalized_keypoint_distance']:.4f}",
        score_status,
        score_color,
        score_note,
    ))

    vis_norm, vis_status, vis_color, vis_note = assess_visibility_overlap(evaluation["visibility_overlap"])
    rows.append(metric_bar_html(
        "Visibility overlap",
        vis_norm,
        f"{100 * evaluation['visibility_overlap']:.1f}%",
        vis_status,
        vis_color,
        vis_note,
    ))

    shared_norm, shared_status, shared_color, shared_note = assess_shared_keypoints(evaluation["shared_keypoints"])
    rows.append(metric_bar_html(
        "Shared normalized keypoints",
        shared_norm,
        f"{evaluation['shared_keypoints']} / 24",
        shared_status,
        shared_color,
        shared_note,
    ))

    scale_norm, scale_status, scale_color, scale_note = assess_scale_ratio(evaluation["bbox_scale_ratio"])
    rows.append(metric_bar_html(
        "BBox scale ratio",
        scale_norm,
        f"{evaluation['bbox_scale_ratio']:.3f}",
        scale_status,
        scale_color,
        scale_note,
    ))

    geom_match = org_record["pose_geometry_label"] == rep_record["pose_geometry_label"]
    geom_norm, geom_status, geom_color, geom_note = assess_boolean_match(
        geom_match,
        "The coarse pose geometry labels match.",
        "The coarse pose geometry labels differ.",
    )
    rows.append(metric_bar_html(
        "Pose geometry label match",
        geom_norm,
        f"{org_record['pose_geometry_label']} vs {rep_record['pose_geometry_label']}",
        geom_status,
        geom_color,
        geom_note,
    ))

    part_match = org_record["visible_parts_signature"] == rep_record["visible_parts_signature"]
    part_norm, part_status, part_color, part_note = assess_boolean_match(
        part_match,
        "Both dogs expose the same body-part signature.",
        "The two dogs expose different visible body parts.",
    )
    rows.append(metric_bar_html(
        "Visible-parts signature match",
        part_norm,
        f"{org_record['visible_parts_signature']} vs {rep_record['visible_parts_signature']}",
        part_status,
        part_color,
        part_note,
    ))

    if warp_diag and warp_diag.get("status") == "ok":
        rmse_norm, rmse_status, rmse_color, rmse_note = assess_relative_rmse(warp_diag["relative_rmse"])
        rows.append(metric_bar_html(
            "Similarity-fit relative RMSE",
            rmse_norm,
            f"{warp_diag['relative_rmse']:.4f}",
            rmse_status,
            rmse_color,
            rmse_note,
        ))
        warp_norm, warp_status, warp_color, warp_note = assess_warp_distortion_ratio(warp_diag["warp_distortion_ratio"])
        rows.append(metric_bar_html(
            "Warp distortion ratio",
            warp_norm,
            f"{warp_diag['warp_distortion_ratio']:.4f}",
            warp_status,
            warp_color,
            warp_note,
        ))

    return render_html_table("Detailed pair-feasibility breakdown", rows)


def render_pair_summary(org_record, rep_record, evaluation):
    pass_color = "#1A7F37" if evaluation["approved_for_01"] else "#D1242F"
    transform_color = "#0969DA" if evaluation["recommended_transform_mode"] == "piecewise_affine" else "#BC4C00"
    html = f'''
    <div style="margin:8px 0 14px 0">
      <div style="font-size:20px;font-weight:700;margin-bottom:8px">Pair feasibility summary</div>
      <div style="margin-bottom:12px">
        {score_chip("Approved", str(evaluation["approved_for_01"]), pass_color)}
        {score_chip("Transform", evaluation["recommended_transform_mode"], transform_color)}
        {score_chip("Primary reason", evaluation["primary_reason"], "#6F42C1")}
      </div>
      <div style="font-size:14px;line-height:1.7">
        <b>Original:</b> {Path(org_record["query_image_path"]).name} | {org_record["pose_geometry_label"]} | {org_record["shape_label"]}<br/>
        <b>Replacement:</b> {Path(rep_record["query_image_path"]).name} | {rep_record["pose_geometry_label"]} | {rep_record["shape_label"]}<br/>
        <b>Warnings:</b> {", ".join(evaluation["warnings"]) if evaluation["warnings"] else "none"}
      </div>
    </div>
    '''
    return HTML(html)


def file_to_data_uri(path: Path):
    mime = "image/png"
    suffix = path.suffix.lower()
    if suffix in {".jpg", ".jpeg"}:
        mime = "image/jpeg"
    elif suffix == ".webp":
        mime = "image/webp"
    encoded = __import__("base64").b64encode(path.read_bytes()).decode("ascii")
    return f"data:{mime};base64,{encoded}"


def save_json(path: Path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


## Segmentation And Cutout-Pose Helpers


In [ ]:
def dog_class_id(model):
    dog_ids = [idx for idx, name in model.names.items() if name == "dog"]
    if not dog_ids:
        raise RuntimeError("The YOLO model does not expose a dog class.")
    return dog_ids[0]


def detect_best_dog(image, conf=0.18):
    results = det_model.predict(image, conf=conf, verbose=False, device=DEVICE)
    result = results[0]
    if result.boxes is None or len(result.boxes) == 0:
        raise RuntimeError("YOLO detector did not return any object boxes.")
    target_cls = dog_class_id(det_model)
    h, w = image.shape[:2]
    dog_candidates = []
    fallback_candidates = []
    for box in result.boxes:
        xyxy = clip_bbox(box.xyxy[0].detach().cpu().numpy().tolist(), w, h)
        score = float(box.conf.item())
        area = bbox_area(xyxy)
        cls_id = int(box.cls.item())
        item = (xyxy, score, area, cls_id)
        fallback_candidates.append(item)
        if cls_id == target_cls:
            dog_candidates.append(item)
    candidates = dog_candidates if dog_candidates else fallback_candidates
    bbox, score, _, cls_id = max(candidates, key=lambda item: item[1] * math.sqrt(item[2]))
    if dog_candidates:
        return bbox, score
    predicted_name = det_model.names.get(cls_id, str(cls_id))
    print(f"YOLO did not label any box as dog; falling back to the strongest object box classified as {predicted_name}.")
    return bbox, score


def crop_with_context(image, bbox, margin_ratio=0.18, min_margin=16):
    h, w = image.shape[:2]
    x1, y1, x2, y2 = [int(round(v)) for v in bbox]
    bw = x2 - x1
    bh = y2 - y1
    margin = int(max(min_margin, round(max(bw, bh) * margin_ratio)))
    crop_bbox = clip_bbox([x1 - margin, y1 - margin, x2 + margin, y2 + margin], w, h)
    cx1, cy1, cx2, cy2 = [int(round(v)) for v in crop_bbox]
    crop = image[cy1:cy2, cx1:cx2].copy()
    local_bbox = [x1 - cx1, y1 - cy1, x2 - cx1, y2 - cy1]
    return crop, [cx1, cy1, cx2, cy2], local_bbox


def uncrop_mask(mask_crop, crop_bbox, full_shape):
    full_mask = np.zeros(full_shape[:2], dtype=bool)
    x1, y1, x2, y2 = crop_bbox
    full_mask[y1:y2, x1:x2] = mask_crop.astype(bool)
    return full_mask


def refine_binary_mask(mask, close_kernel=9, open_kernel=3):
    mask_u8 = mask.astype(np.uint8)
    if close_kernel > 0:
        kernel = np.ones((close_kernel, close_kernel), np.uint8)
        mask_u8 = cv2.morphologyEx(mask_u8, cv2.MORPH_CLOSE, kernel)
    if open_kernel > 0:
        kernel = np.ones((open_kernel, open_kernel), np.uint8)
        mask_u8 = cv2.morphologyEx(mask_u8, cv2.MORPH_OPEN, kernel)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    if num_labels <= 1:
        return mask_u8.astype(bool)
    keep = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    return labels == keep


def estimate_foreground_mask_from_borders(image, pad_ratio=0.04):
    h, w = image.shape[:2]
    pad_y = max(4, int(round(h * pad_ratio)))
    pad_x = max(4, int(round(w * pad_ratio)))
    border_pixels = np.concatenate(
        [
            image[:pad_y].reshape(-1, 3),
            image[-pad_y:].reshape(-1, 3),
            image[:, :pad_x].reshape(-1, 3),
            image[:, -pad_x:].reshape(-1, 3),
        ],
        axis=0,
    ).astype(np.float32)
    bg_color = np.median(border_pixels, axis=0)
    diff = np.linalg.norm(image.astype(np.float32) - bg_color[None, None, :], axis=2)
    threshold = max(18.0, float(np.percentile(diff, 72)))
    mask = diff > threshold
    return refine_binary_mask(mask, close_kernel=9, open_kernel=3)


def segment_with_yolo_seg(image, bbox, conf=0.12, context_ratio=0.22):
    crop, crop_bbox, local_bbox = crop_with_context(image, bbox, margin_ratio=context_ratio)
    results = seg_model.predict(crop, conf=conf, verbose=False, device=DEVICE)
    result = results[0]
    if result.masks is None or result.boxes is None or len(result.boxes) == 0:
        return None
    target_cls = dog_class_id(seg_model)
    masks = result.masks.data.detach().cpu().numpy()
    h, w = crop.shape[:2]
    dog_candidates = []
    fallback_candidates = []
    for i, box in enumerate(result.boxes):
        mask = masks[i]
        if mask.shape != (h, w):
            mask = cv2.resize(mask.astype(np.float32), (w, h), interpolation=cv2.INTER_LINEAR)
        mask_bool = mask > 0.45
        if mask_bool.sum() < 20:
            continue
        local_box = clip_bbox(box.xyxy[0].detach().cpu().numpy().tolist(), w, h)
        focus = bbox_iou(local_box, local_bbox)
        full_mask = uncrop_mask(mask_bool, crop_bbox, image.shape)
        cls_id = int(box.cls.item())
        item = (full_mask, float(box.conf.item()), focus, cls_id)
        fallback_candidates.append(item)
        if cls_id == target_cls:
            dog_candidates.append(item)
    candidates = dog_candidates if dog_candidates else fallback_candidates
    if not candidates:
        return None
    best_mask, _, _, _ = max(candidates, key=lambda item: item[0].sum() * (0.3 + item[1]) * (0.2 + item[2]))
    return refine_binary_mask(best_mask, close_kernel=9, open_kernel=3)


def score_mask_with_prior(mask, prior_mask):
    inter = np.logical_and(mask, prior_mask).sum()
    outside = np.logical_and(mask, ~prior_mask).sum()
    prior_area = max(1, prior_mask.sum())
    mask_area = max(1, mask.sum())
    return inter / prior_area + inter / mask_area - 0.75 * outside / mask_area


def segment_with_sam(image, bbox, prior_mask=None, context_ratio=0.22, max_side=1280):
    crop, crop_bbox, local_bbox = crop_with_context(image, bbox, margin_ratio=context_ratio)
    h, w = crop.shape[:2]
    scale = min(1.0, max_side / max(h, w))
    if scale < 1.0:
        infer_w = int(round(w * scale))
        infer_h = int(round(h * scale))
        infer_image = cv2.resize(crop, (infer_w, infer_h), interpolation=cv2.INTER_AREA)
        infer_bbox = [int(round(v * scale)) for v in local_bbox]
    else:
        infer_image = crop
        infer_bbox = local_bbox
    results = sam_model.predict(infer_image, bboxes=[infer_bbox], verbose=False, device=DEVICE)
    result = results[0]
    if result.masks is None or len(result.masks.data) == 0:
        return None
    masks = result.masks.data.detach().cpu().numpy()
    candidates = []
    for mask in masks:
        if mask.shape != infer_image.shape[:2]:
            mask = cv2.resize(mask.astype(np.float32), (infer_image.shape[1], infer_image.shape[0]), interpolation=cv2.INTER_NEAREST)
        if scale < 1.0:
            mask = cv2.resize(mask.astype(np.float32), (w, h), interpolation=cv2.INTER_NEAREST)
        full_mask = uncrop_mask(mask > 0.5, crop_bbox, image.shape)
        candidates.append(full_mask)
    if not candidates:
        return None
    if prior_mask is not None:
        best = max(candidates, key=lambda item: score_mask_with_prior(item, prior_mask))
        prior_soft = cv2.dilate(prior_mask.astype(np.uint8), np.ones((17, 17), np.uint8), iterations=1).astype(bool)
        constrained = np.logical_and(best, prior_soft)
        if constrained.sum() > 0.35 * max(1, prior_mask.sum()):
            best = constrained
    else:
        x1, y1, x2, y2 = [int(round(v)) for v in bbox]
        best = max(candidates, key=lambda item: int(item[y1:y2, x1:x2].sum()))
    return refine_binary_mask(best, close_kernel=7, open_kernel=3)


def refine_mask_grabcut(image, mask, iter_count=5):
    if mask is None or mask.sum() < 10:
        return mask
    mask_u8 = mask.astype(np.uint8)
    eroded = cv2.erode(mask_u8, np.ones((5, 5), np.uint8), iterations=1)
    dilated = cv2.dilate(mask_u8, np.ones((21, 21), np.uint8), iterations=1)
    gc_mask = np.full(mask.shape, cv2.GC_BGD, dtype=np.uint8)
    gc_mask[dilated > 0] = cv2.GC_PR_FGD
    gc_mask[eroded > 0] = cv2.GC_FGD
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)
    bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    try:
        cv2.grabCut(bgr, gc_mask, None, bgd_model, fgd_model, iter_count, cv2.GC_INIT_WITH_MASK)
        refined = np.logical_or(gc_mask == cv2.GC_FGD, gc_mask == cv2.GC_PR_FGD)
        if refined.sum() > 0.25 * mask.sum():
            return refine_binary_mask(refined, close_kernel=7, open_kernel=3)
    except Exception as exc:
        print("GrabCut refinement skipped:", type(exc).__name__, exc)
    return refine_binary_mask(mask, close_kernel=7, open_kernel=3)


def build_consensus_mask(mask_a, mask_b):
    if mask_a is None or mask_b is None:
        return None
    overlap = np.logical_and(mask_a, mask_b)
    if overlap.sum() >= 0.35 * min(mask_a.sum(), mask_b.sum()):
        return refine_binary_mask(np.logical_or(mask_a, mask_b), close_kernel=9, open_kernel=3)
    return None


def edge_touch_fraction(mask, width=3):
    return float(mask[:width].mean() + mask[-width:].mean() + mask[:, :width].mean() + mask[:, -width:].mean())


def mask_keypoint_coverage(mask, pose_record, slack=6):
    if pose_record is None:
        return 0.0
    h, w = mask.shape[:2]
    covered = 0
    total = 0
    for pt, conf in zip(pose_record["points"], pose_record["confs"]):
        if conf < KEYPOINT_CONF:
            continue
        total += 1
        x = int(round(pt[0]))
        y = int(round(pt[1]))
        x1 = max(0, x - slack)
        y1 = max(0, y - slack)
        x2 = min(w, x + slack + 1)
        y2 = min(h, y + slack + 1)
        if x1 < x2 and y1 < y2 and mask[y1:y2, x1:x2].any():
            covered += 1
    return covered / max(1, total)


REP_MASK_INSET_KERNEL = 5
REP_MASK_INSET_ITERATIONS = 1
REP_MASK_INSET_MIN_AREA_RATIO = 0.72
REP_CORE_MASK_KERNEL = 13
REP_ALPHA_FEATHER_RADIUS = 6.0
REP_EDGE_BG_RING_KERNEL = 19
REP_EDGE_DECONTAM_STRENGTH = 0.92
REP_EDGE_ALPHA_FLOOR = 0.35


def inset_mask_inward(
    mask,
    kernel_size=REP_MASK_INSET_KERNEL,
    iterations=REP_MASK_INSET_ITERATIONS,
    min_area_ratio=REP_MASK_INSET_MIN_AREA_RATIO,
):
    if mask is None or mask.sum() < 10 or kernel_size <= 1 or iterations <= 0:
        return mask
    kernel = np.ones((int(kernel_size), int(kernel_size)), np.uint8)
    inset = cv2.erode(mask.astype(np.uint8), kernel, iterations=int(iterations)) > 0
    min_pixels = max(25, int(round(float(mask.sum()) * float(min_area_ratio))))
    if inset.sum() < min_pixels:
        return mask.copy()
    return refine_binary_mask(inset, close_kernel=3, open_kernel=0)


def build_mask_layers(crop_mask, core_kernel=REP_CORE_MASK_KERNEL, feather_radius=REP_ALPHA_FEATHER_RADIUS):
    crop_mask = crop_mask.astype(bool)
    if crop_mask.sum() < 10:
        alpha = crop_mask.astype(np.float32)
        return crop_mask.copy(), np.zeros_like(crop_mask, dtype=bool), alpha
    core_mask = inset_mask_inward(
        crop_mask,
        kernel_size=core_kernel,
        iterations=1,
        min_area_ratio=0.45,
    )
    if core_mask.sum() < max(25, int(round(crop_mask.sum() * 0.35))):
        core_mask = crop_mask.copy()
    boundary_ring = np.logical_and(crop_mask, ~core_mask)
    dist_in = cv2.distanceTransform(crop_mask.astype(np.uint8), cv2.DIST_L2, 5).astype(np.float32)
    alpha = np.clip(dist_in / max(1.0, float(feather_radius)), 0.0, 1.0)
    alpha[core_mask] = 1.0
    alpha[~crop_mask] = 0.0
    return core_mask, boundary_ring, alpha


def estimate_background_rgb_around_mask(crop_rgb, crop_mask, ring_kernel=REP_EDGE_BG_RING_KERNEL):
    kernel = np.ones((int(ring_kernel), int(ring_kernel)), np.uint8)
    outer_ring = np.logical_and(
        cv2.dilate(crop_mask.astype(np.uint8), kernel, iterations=1).astype(bool),
        ~crop_mask,
    )
    if outer_ring.sum() < 20:
        h, w = crop_mask.shape
        border = np.zeros_like(crop_mask, dtype=bool)
        border[:2] = True
        border[-2:] = True
        border[:, :2] = True
        border[:, -2:] = True
        outer_ring = border
    bg_pixels = crop_rgb[outer_ring]
    if bg_pixels.size == 0:
        bg_pixels = crop_rgb.reshape(-1, 3)
    return np.median(bg_pixels.astype(np.float32), axis=0)


def decontaminate_boundary_rgb(
    crop_rgb,
    crop_mask,
    alpha,
    boundary_ring,
    strength=REP_EDGE_DECONTAM_STRENGTH,
    alpha_floor=REP_EDGE_ALPHA_FLOOR,
):
    if boundary_ring.sum() < 10:
        return crop_rgb.copy(), None
    bg_rgb = estimate_background_rgb_around_mask(crop_rgb, crop_mask)
    rgb = crop_rgb.astype(np.float32)
    alpha3 = alpha[..., None].astype(np.float32)
    safe_alpha = np.clip(alpha3, float(alpha_floor), 1.0)
    recovered = (rgb - (1.0 - alpha3) * bg_rgb.reshape(1, 1, 3)) / safe_alpha
    recovered = np.clip(recovered, 0.0, 255.0)
    ring_weight = boundary_ring[..., None].astype(np.float32) * np.clip((1.0 - alpha3) * 1.35 + 0.15, 0.0, 1.0)
    weight = np.clip(float(strength) * ring_weight, 0.0, 1.0)
    decontaminated = rgb * (1.0 - weight) + recovered * weight
    decontaminated[~crop_mask] = rgb[~crop_mask]
    return np.clip(decontaminated, 0.0, 255.0).astype(np.uint8), bg_rgb


def local_mask_to_full(local_mask, bbox, full_shape):
    full_mask = np.zeros(full_shape[:2], dtype=bool)
    x1, y1, x2, y2 = [int(v) for v in bbox]
    full_mask[y1:y2, x1:x2] = local_mask.astype(bool)
    return full_mask


def extract_rgba(image, mask, return_details=False, decontaminate_edges=False):
    bbox = mask_to_bbox(mask)
    if bbox is None:
        raise RuntimeError("Cannot extract RGBA from an empty mask.")
    x1, y1, x2, y2 = bbox
    crop_rgb = image[y1:y2, x1:x2].copy()
    crop_mask = mask[y1:y2, x1:x2].copy().astype(bool)
    core_mask, boundary_ring, alpha = build_mask_layers(crop_mask)
    bg_rgb = None
    crop_rgb_out = crop_rgb.copy()
    if decontaminate_edges:
        crop_rgb_out, bg_rgb = decontaminate_boundary_rgb(crop_rgb_out, crop_mask, alpha, boundary_ring)
    rgba = np.dstack([crop_rgb_out, np.uint8(np.clip(alpha, 0.0, 1.0) * 255)])
    if return_details:
        return {
            "rgb": crop_rgb_out,
            "mask": crop_mask,
            "rgba": rgba,
            "bbox": bbox,
            "core_mask": core_mask,
            "boundary_ring": boundary_ring,
            "alpha": alpha,
            "edge_background_rgb": None if bg_rgb is None else [float(v) for v in bg_rgb],
        }
    return crop_rgb_out, crop_mask, rgba, bbox


def detect_pose_record(image):
    results = pose_model.predict(image, conf=POSE_CONF, imgsz=POSE_IMGSZ, device=DEVICE, verbose=False)
    result = results[0]
    best_idx = choose_best_pose_instance(result)
    if best_idx is None:
        raise RuntimeError("Pose model did not return a usable dog instance.")
    bbox = clip_bbox(result.boxes[best_idx].xyxy[0].detach().cpu().numpy().tolist(), image.shape[1], image.shape[0])
    keypoints_xy, keypoints_conf = get_keypoint_arrays(result, best_idx)
    return {
        "bbox_xyxy": bbox,
        "detection_conf": round(float(result.boxes[best_idx].conf.item()), 6),
        "visible_keypoints": int((keypoints_conf >= KEYPOINT_CONF).sum()),
        "orientation": orientation_label(keypoints_xy, keypoints_conf),
        "view_class": classify_view(keypoints_xy, keypoints_conf, bbox),
        "points": keypoints_xy,
        "confs": keypoints_conf,
    }


def build_pose_canvas(crop_rgb, crop_mask, pad=40):
    h, w = crop_rgb.shape[:2]
    canvas = np.full((h + 2 * pad, w + 2 * pad, 3), 255, dtype=np.uint8)
    masked_crop = crop_rgb.copy()
    masked_crop[~crop_mask] = 255
    canvas[pad:pad + h, pad:pad + w] = masked_crop
    return canvas, pad, pad


def translate_pose_record_local(pose_record, dx=0.0, dy=0.0):
    translated = {
        **pose_record,
        "bbox_xyxy": [
            int(round(pose_record["bbox_xyxy"][0] + dx)),
            int(round(pose_record["bbox_xyxy"][1] + dy)),
            int(round(pose_record["bbox_xyxy"][2] + dx)),
            int(round(pose_record["bbox_xyxy"][3] + dy)),
        ],
        "points": pose_record["points"].copy(),
        "confs": pose_record["confs"].copy(),
    }
    translated["points"][:, 0] += float(dx)
    translated["points"][:, 1] += float(dy)
    return translated


def mirror_pose_record_x(pose_record, width):
    mirrored = {
        **pose_record,
        "bbox_xyxy": [
            int(round((width - 1) - pose_record["bbox_xyxy"][2])),
            pose_record["bbox_xyxy"][1],
            int(round((width - 1) - pose_record["bbox_xyxy"][0])),
            pose_record["bbox_xyxy"][3],
        ],
        "points": pose_record["points"].copy(),
        "confs": pose_record["confs"].copy(),
    }
    mirrored["points"][:, 0] = (width - 1) - mirrored["points"][:, 0]
    return mirrored


def shared_keypoints_from_local(orig_record, repl_local_record):
    names = []
    src_pts = []
    dst_pts = []
    for idx, name in enumerate(DOG_KEYPOINT_NAMES):
        if orig_record["confs"][idx] >= KEYPOINT_CONF and repl_local_record["confs"][idx] >= KEYPOINT_CONF:
            names.append(name)
            src_pts.append(repl_local_record["points"][idx])
            dst_pts.append(orig_record["points"][idx])
    if not src_pts:
        return [], np.zeros((0, 2), dtype=np.float32), np.zeros((0, 2), dtype=np.float32)
    return names, np.array(src_pts, dtype=np.float32), np.array(dst_pts, dtype=np.float32)


def pose_rmse(src_pts, dst_pts, tform):
    predicted = tform(src_pts)
    return float(np.sqrt(np.mean(np.sum((predicted - dst_pts) ** 2, axis=1))))


def build_similarity_seed(src_pts, dst_pts):
    tform = SimilarityTransform()
    if len(src_pts) >= 2 and tform.estimate(src_pts, dst_pts):
        return tform
    raise RuntimeError("Failed to estimate initial similarity transform from shared keypoints.")


def build_piecewise_or_similarity(src_pts, dst_pts, source_shape):
    similarity = build_similarity_seed(src_pts, dst_pts)
    h, w = source_shape[:2]
    src_corners = np.array([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]], dtype=np.float32)
    dst_corners = similarity(src_corners)
    if len(src_pts) >= 4:
        tform = PiecewiseAffineTransform()
        ok = tform.estimate(np.vstack([src_pts, src_corners]), np.vstack([dst_pts, dst_corners]))
        if ok:
            return tform, "piecewise_affine"
    return similarity, "similarity_only"


def transform_pose_record(pose_record, tform):
    points = tform(pose_record["points"])
    x1, y1, x2, y2 = pose_record["bbox_xyxy"]
    corners = np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]], dtype=np.float32)
    warped_corners = tform(corners)
    bbox = [
        int(round(warped_corners[:, 0].min())),
        int(round(warped_corners[:, 1].min())),
        int(round(warped_corners[:, 0].max())),
        int(round(warped_corners[:, 1].max())),
    ]
    return {
        **pose_record,
        "points": np.array(points, dtype=np.float32),
        "bbox_xyxy": bbox,
        "confs": pose_record["confs"].copy(),
    }


def naive_transform_pose_record(pose_record, target_bbox, source_shape):
    x1, y1, x2, y2 = target_bbox
    target_w = max(1, x2 - x1)
    target_h = max(1, y2 - y1)
    src_h, src_w = source_shape[:2]
    scale = min(target_w / max(1, src_w), target_h / max(1, src_h))
    new_w = max(1, int(round(src_w * scale)))
    new_h = max(1, int(round(src_h * scale)))
    left = float((x1 + x2) / 2.0 - new_w / 2.0)
    top = float(y2 - new_h)

    points = pose_record["points"].copy()
    points[:, 0] = points[:, 0] * scale + left
    points[:, 1] = points[:, 1] * scale + top
    bbox = [
        int(round(pose_record["bbox_xyxy"][0] * scale + left)),
        int(round(pose_record["bbox_xyxy"][1] * scale + top)),
        int(round(pose_record["bbox_xyxy"][2] * scale + left)),
        int(round(pose_record["bbox_xyxy"][3] * scale + top)),
    ]
    return {
        **pose_record,
        "points": points,
        "bbox_xyxy": bbox,
        "confs": pose_record["confs"].copy(),
    }


## Self-Contained Pair Analysis


In [ ]:
def analyze_segmentation(image, bbox, prior_pose=None, is_replacement=False):
    prior_mask = estimate_foreground_mask_from_borders(image) if is_replacement else None
    yolo_mask = segment_with_yolo_seg(image, bbox)
    sam_mask = segment_with_sam(image, bbox, prior_mask=prior_mask)
    consensus_mask = build_consensus_mask(yolo_mask, sam_mask)

    candidates = []
    if is_replacement:
        candidates.append(("Foreground prior", prior_mask))
    if yolo_mask is not None:
        candidates.append(("YOLO-seg", yolo_mask))
    if sam_mask is not None:
        candidates.append(("SAM", sam_mask))
    if consensus_mask is not None:
        candidates.append(("Consensus union", consensus_mask))
    if not candidates:
        raise RuntimeError("No usable segmentation candidates were produced.")

    def mask_score(mask):
        local_box = mask_to_bbox(mask)
        if local_box is None:
            return -1e9
        keypoint_cov = mask_keypoint_coverage(mask, prior_pose, slack=7) if prior_pose is not None else 0.0
        area_ratio = mask.sum() / max(1, bbox_area(bbox))
        area_penalty = abs(area_ratio - 1.0)
        score = 1.7 * bbox_iou(local_box, bbox) + 1.2 * keypoint_cov - 0.2 * area_penalty - 0.25 * edge_touch_fraction(mask)
        if is_replacement and prior_mask is not None:
            prior_iou = np.logical_and(mask, prior_mask).sum() / max(1, np.logical_or(mask, prior_mask).sum())
            score += 1.0 * prior_iou
        return score

    source_name, mask_raw = max(candidates, key=lambda item: mask_score(item[1]))
    refined_mask = refine_mask_grabcut(image, mask_raw, iter_count=4 if is_replacement else 5)
    inset_mask = inset_mask_inward(refined_mask) if is_replacement else refined_mask
    return {
        "source_name": source_name,
        "prior_mask": prior_mask,
        "yolo_mask": yolo_mask,
        "sam_mask": sam_mask,
        "consensus_mask": consensus_mask,
        "refined_mask": refined_mask,
        "inset_mask": inset_mask if is_replacement else None,
        "mask": inset_mask,
    }


def build_cutout_pose_analysis(original_img, replacement_img, orig_bbox, repl_bbox, orig_global_pose, repl_global_pose):
    orig_seg = analyze_segmentation(original_img, orig_bbox, prior_pose={
        "points": np.array(orig_global_pose["keypoints_xy"], dtype=np.float32),
        "confs": np.array(orig_global_pose["keypoints_conf"], dtype=np.float32),
    }, is_replacement=False)
    repl_seg = analyze_segmentation(replacement_img, repl_bbox, prior_pose={
        "points": np.array(repl_global_pose["keypoints_xy"], dtype=np.float32),
        "confs": np.array(repl_global_pose["keypoints_conf"], dtype=np.float32),
    }, is_replacement=True)

    org_dog_rgb, org_dog_mask, org_dog_rgba, org_dog_bbox = extract_rgba(original_img, orig_seg["mask"])
    repl_extract = extract_rgba(replacement_img, repl_seg["mask"], return_details=True, decontaminate_edges=True)
    repl_fg_rgb = repl_extract["rgb"]
    repl_fg_mask = repl_extract["mask"]
    repl_fg_rgba = repl_extract["rgba"]
    repl_fg_bbox = repl_extract["bbox"]
    replacement_core_mask_full = local_mask_to_full(repl_extract["core_mask"], repl_fg_bbox, replacement_img.shape)
    replacement_boundary_ring_full = local_mask_to_full(repl_extract["boundary_ring"], repl_fg_bbox, replacement_img.shape)

    org_pose_canvas, org_pad_x, org_pad_y = build_pose_canvas(org_dog_rgb, org_dog_mask)
    rep_pose_canvas, rep_pad_x, rep_pad_y = build_pose_canvas(repl_fg_rgb, repl_fg_mask)

    org_cutout_pose_canvas = detect_pose_record(org_pose_canvas)
    rep_cutout_pose_canvas = detect_pose_record(rep_pose_canvas)
    org_cutout_pose = translate_pose_record_local(org_cutout_pose_canvas, dx=-org_pad_x, dy=-org_pad_y)
    rep_cutout_pose = translate_pose_record_local(rep_cutout_pose_canvas, dx=-rep_pad_x, dy=-rep_pad_y)

    return {
        "original_segmentation": orig_seg,
        "replacement_segmentation": repl_seg,
        "original_crop_bbox": org_dog_bbox,
        "replacement_crop_bbox": repl_fg_bbox,
        "original_rgba": org_dog_rgba,
        "replacement_rgba": repl_fg_rgba,
        "replacement_core_mask": replacement_core_mask_full,
        "replacement_boundary_ring": replacement_boundary_ring_full,
        "replacement_edge_background_rgb": repl_extract["edge_background_rgb"],
        "original_pose_canvas": org_pose_canvas,
        "replacement_pose_canvas": rep_pose_canvas,
        "original_cutout_pose": org_cutout_pose,
        "replacement_cutout_pose": rep_cutout_pose,
        "replacement_crop_shape": repl_fg_rgba.shape,
    }


def build_warp_diagnostics(original_img, cutout_result, evaluation):
    org_local = cutout_result["original_cutout_pose"]
    rep_local = cutout_result["replacement_cutout_pose"]
    org_crop_bbox = cutout_result["original_crop_bbox"]
    orig_pose = translate_pose_record_local(org_local, dx=org_crop_bbox[0], dy=org_crop_bbox[1])

    rep_rgba = cutout_result["replacement_rgba"]
    rep_canonical = rep_local
    mirror_replacement = (
        orig_pose["view_class"] == "side_profile"
        and rep_local["view_class"] == "side_profile"
        and orig_pose["orientation"] in {"left", "right"}
        and rep_local["orientation"] in {"left", "right"}
        and orig_pose["orientation"] != rep_local["orientation"]
    )
    if mirror_replacement:
        rep_canonical = mirror_pose_record_x(rep_local, rep_rgba.shape[1])

    shared_names, src_pts, dst_pts = shared_keypoints_from_local(orig_pose, rep_canonical)
    if len(shared_names) < 3:
        return {
            "status": "too_few_shared_keypoints",
            "shared_keypoints": len(shared_names),
            "shared_keypoint_names": shared_names,
        }

    similarity_seed = build_similarity_seed(src_pts, dst_pts)
    similarity_pose = transform_pose_record(rep_canonical, similarity_seed)
    similarity_rmse = pose_rmse(src_pts, dst_pts, similarity_seed)
    relative_rmse = similarity_rmse / max(1.0, bbox_diag(orig_pose["bbox_xyxy"]))

    recommended_mode = evaluation["recommended_transform_mode"]
    allow_piecewise = len(shared_names) >= 5 and relative_rmse <= 0.12 and recommended_mode == "piecewise_affine"
    force_naive = relative_rmse > 0.24

    if force_naive:
        final_mode = "naive_box_fallback"
        final_pose = naive_transform_pose_record(rep_canonical, orig_pose["bbox_xyxy"], rep_rgba.shape)
    else:
        if allow_piecewise:
            tform, final_mode = build_piecewise_or_similarity(src_pts, dst_pts, rep_rgba.shape)
        else:
            tform, final_mode = similarity_seed, "similarity_only"
        final_pose = similarity_pose if final_mode == "similarity_only" else transform_pose_record(rep_canonical, tform)

    valid = (rep_canonical["confs"] >= KEYPOINT_CONF) & (orig_pose["confs"] >= KEYPOINT_CONF)
    if valid.any():
        final_vs_seed = float(np.mean(np.linalg.norm(final_pose["points"][valid] - similarity_pose["points"][valid], axis=1)))
        final_to_target = float(np.mean(np.linalg.norm(final_pose["points"][valid] - orig_pose["points"][valid], axis=1)))
    else:
        final_vs_seed = 0.0
        final_to_target = 0.0

    warp_distortion_ratio = final_vs_seed / max(1.0, bbox_diag(orig_pose["bbox_xyxy"]))
    target_fit_ratio = final_to_target / max(1.0, bbox_diag(orig_pose["bbox_xyxy"]))

    return {
        "status": "ok",
        "mirror_replacement": mirror_replacement,
        "shared_keypoints": len(shared_names),
        "shared_keypoint_names": shared_names,
        "similarity_rmse": float(similarity_rmse),
        "relative_rmse": float(relative_rmse),
        "warp_distortion_ratio": float(warp_distortion_ratio),
        "target_fit_ratio": float(target_fit_ratio),
        "recommended_transform_mode": recommended_mode,
        "final_transform_mode": final_mode,
        "target_pose": orig_pose,
        "similarity_seed_pose": similarity_pose,
        "final_pose": final_pose,
        "original_image": original_img,
    }


def plot_warp_overlay(diagnostics, figsize=(10, 8)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(diagnostics["original_image"])
    draw_pose_on_axis(ax, diagnostics["target_pose"], color="#2DA44E", alpha=0.95, linewidth=2.4, show_bbox=True, linestyle="-")
    draw_pose_on_axis(ax, diagnostics["similarity_seed_pose"], color="#FB8500", alpha=0.35, linewidth=2.0, show_bbox=True, linestyle="--")
    draw_pose_on_axis(ax, diagnostics["final_pose"], color="#C026D3", alpha=0.88, linewidth=2.2, show_bbox=True, linestyle="-")
    ax.set_title(
        f"Pose-fitting overlay | final={diagnostics['final_transform_mode']} | "
        f"relative RMSE={diagnostics['relative_rmse']:.4f}"
    )
    ax.axis("off")
    legend_lines = [
        plt.Line2D([0], [0], color="#2DA44E", lw=2.5, label="Target pose"),
        plt.Line2D([0], [0], color="#FB8500", lw=2.5, ls="--", alpha=0.5, label="Similarity seed"),
        plt.Line2D([0], [0], color="#C026D3", lw=2.5, label="Final fitted pose"),
    ]
    ax.legend(handles=legend_lines, loc="upper right", frameon=True)
    plt.tight_layout()
    return fig


def build_segmentation_preview_figure(original_img, replacement_img, orig_bbox, repl_bbox, cutout_result):
    orig_seg = cutout_result["original_segmentation"]
    repl_seg = cutout_result["replacement_segmentation"]
    items = [
        ("Original detection", draw_bbox(original_img, orig_bbox)),
        ("Original refined mask", overlay_mask(original_img, orig_seg["mask"], color=(255, 180, 0))),
        ("Replacement detection", draw_bbox(replacement_img, repl_bbox)),
        ("Replacement refined mask", overlay_mask(replacement_img, repl_seg["refined_mask"], color=(255, 80, 80))),
        ("Replacement inset mask (used)", overlay_mask(replacement_img, repl_seg["mask"], color=(80, 220, 255))),
        ("Replacement boundary ring", overlay_mask(replacement_img, cutout_result["replacement_boundary_ring"], color=(255, 0, 255))),
        ("Original pose canvas", cutout_result["original_pose_canvas"]),
        ("Replacement pose canvas", cutout_result["replacement_pose_canvas"]),
        ("Original cutout RGBA", rgba_on_checkerboard(cutout_result["original_rgba"])),
        ("Replacement cutout RGBA", rgba_on_checkerboard(cutout_result["replacement_rgba"])),
    ]
    return plot_images(items, cols=3, figsize=(18, 15))


def run_pair_analysis_from_paths(org_path: Path, rep_path: Path, display_results=True):
    org_record = query_pose_record(org_path, pose_model)
    rep_record = query_pose_record(rep_path, pose_model)
    if org_record.get("status") != "ok":
        raise RuntimeError("No dog pose could be extracted from the original image.")
    if rep_record.get("status") != "ok":
        raise RuntimeError("No dog pose could be extracted from the replacement image.")

    original_img = read_rgb(org_path)
    replacement_img = read_rgb(rep_path)
    orig_bbox, _ = detect_best_dog(original_img)
    repl_bbox, _ = detect_best_dog(replacement_img)

    evaluation = evaluate_pair(org_record, rep_record)
    cutout_result = build_cutout_pose_analysis(original_img, replacement_img, orig_bbox, repl_bbox, org_record, rep_record)
    warp_diag = build_warp_diagnostics(original_img, cutout_result, evaluation)

    pair_name = f"{org_path.stem}__{rep_path.stem}"
    pair_dir = OUTPUT_ROOT / pair_name
    pair_dir.mkdir(parents=True, exist_ok=True)

    dual_fig = render_dual_pose_figure(org_record, rep_record)
    normalized_fig = plot_normalized_pose_overlay(org_record, rep_record, figsize=(7, 7))
    segmentation_fig = build_segmentation_preview_figure(original_img, replacement_img, orig_bbox, repl_bbox, cutout_result)
    dual_fig.savefig(pair_dir / "pair_pose_visualization.png", dpi=160, bbox_inches="tight")
    normalized_fig.savefig(pair_dir / "normalized_pose_overlay.png", dpi=160, bbox_inches="tight")
    segmentation_fig.savefig(pair_dir / "segmentation_and_cutout_preview.png", dpi=160, bbox_inches="tight")

    warp_fig = None
    if warp_diag.get("status") == "ok":
        warp_fig = plot_warp_overlay(warp_diag, figsize=(10, 8))
        warp_fig.savefig(pair_dir / "pose_fitting_overlay.png", dpi=160, bbox_inches="tight")

    summary = {
        "pair_name": pair_name,
        "original_path": str(org_path),
        "replacement_path": str(rep_path),
        "original_record": {k: v for k, v in org_record.items() if k not in {"keypoints_xy", "keypoints_conf", "normalized_keypoints"}},
        "replacement_record": {k: v for k, v in rep_record.items() if k not in {"keypoints_xy", "keypoints_conf", "normalized_keypoints"}},
        "original_keypoints_xy": org_record["keypoints_xy"],
        "replacement_keypoints_xy": rep_record["keypoints_xy"],
        "original_normalized_keypoints": org_record["normalized_keypoints"],
        "replacement_normalized_keypoints": rep_record["normalized_keypoints"],
        "evaluation": evaluation,
        "segmentation_summary": {
            "original": {
                "mask_source": cutout_result["original_segmentation"]["source_name"],
                "mask_area": int(cutout_result["original_segmentation"]["mask"].sum()),
            },
            "replacement": {
                "mask_source": cutout_result["replacement_segmentation"]["source_name"],
                "refined_area": int(cutout_result["replacement_segmentation"]["refined_mask"].sum()),
                "inset_area": int(cutout_result["replacement_segmentation"]["mask"].sum()),
                "core_area": int(cutout_result["replacement_core_mask"].sum()),
                "boundary_ring_area": int(cutout_result["replacement_boundary_ring"].sum()),
                "inset_kernel": REP_MASK_INSET_KERNEL,
                "inset_iterations": REP_MASK_INSET_ITERATIONS,
                "core_kernel": REP_CORE_MASK_KERNEL,
                "edge_decontam_strength": REP_EDGE_DECONTAM_STRENGTH,
                "edge_background_rgb": cutout_result["replacement_edge_background_rgb"],
            },
        },
        "cutout_pose_summary": {
            "original_cutout_pose": {
                "view_class": cutout_result["original_cutout_pose"]["view_class"],
                "orientation": cutout_result["original_cutout_pose"]["orientation"],
                "visible_keypoints": int(cutout_result["original_cutout_pose"]["visible_keypoints"]),
                "bbox_xyxy": [int(v) for v in cutout_result["original_cutout_pose"]["bbox_xyxy"]],
            },
            "replacement_cutout_pose": {
                "view_class": cutout_result["replacement_cutout_pose"]["view_class"],
                "orientation": cutout_result["replacement_cutout_pose"]["orientation"],
                "visible_keypoints": int(cutout_result["replacement_cutout_pose"]["visible_keypoints"]),
                "bbox_xyxy": [int(v) for v in cutout_result["replacement_cutout_pose"]["bbox_xyxy"]],
            },
        },
        "warp_diagnostics": {
            key: value
            for key, value in warp_diag.items()
            if key not in {"target_pose", "similarity_seed_pose", "final_pose", "original_image"}
        },
    }
    save_json(pair_dir / "pair_feasibility_summary.json", summary)

    if display_results:
        display(render_pair_summary(org_record, rep_record, evaluation))
        display(dual_fig)
        display(normalized_fig)
        display(segmentation_fig)
        plt.close(dual_fig)
        plt.close(normalized_fig)
        plt.close(segmentation_fig)
        display(render_pair_metric_table(org_record, rep_record, evaluation, warp_diag=warp_diag))
        if warp_fig is not None:
            display(HTML(
                "<div style='margin:10px 0 8px 0'><h3 style='margin:0 0 8px 0'>Pose-fitting distortion view</h3>"
                "<div style='font-size:14px'>The orange skeleton is the similarity seed, while the purple skeleton is the final fitted pose. "
                "This makes the fitting distortion visually explicit.</div></div>"
            ))
            display(warp_fig)
            plt.close(warp_fig)
        else:
            display(HTML(f"<div style='margin:10px 0 8px 0;color:#D1242F'>Warp overlay unavailable: {warp_diag.get('status')}.</div>"))
        display(HTML(f"<div style='margin-top:10px;font-size:14px'>Saved outputs to <code>{pair_dir}</code></div>"))
    else:
        plt.close(dual_fig)
        plt.close(normalized_fig)
        plt.close(segmentation_fig)
        if warp_fig is not None:
            plt.close(warp_fig)
    return {
        "org_record": org_record,
        "rep_record": rep_record,
        "evaluation": evaluation,
        "cutout_result": cutout_result,
        "warp_diagnostics": warp_diag,
        "output_dir": str(pair_dir),
    }


## Interactive Pair Analysis

Upload one original dog image and one replacement-dog image, then click **Analyze pair**.


In [ ]:
status_output = widgets.Output()
result_output = widgets.Output()

org_upload = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="Upload original image",
)
rep_upload = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="Upload replacement image",
)
analyze_button = widgets.Button(
    description="Analyze pair",
    button_style="primary",
    icon="check",
)


def on_analyze_pair(_):
    with status_output:
        clear_output(wait=True)
        print("Running self-contained pair analysis...")
    with result_output:
        clear_output(wait=True)

    org_entry = extract_upload_entry(org_upload.value)
    rep_entry = extract_upload_entry(rep_upload.value)
    if org_entry is None or rep_entry is None:
        with status_output:
            clear_output(wait=True)
            print("Please upload both the original image and the replacement image.")
        return

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    upload_dir = UPLOAD_ROOT / timestamp
    org_path = save_upload_entry(org_entry, upload_dir, "org")
    rep_path = save_upload_entry(rep_entry, upload_dir, "rep")

    with result_output:
        run_pair_analysis_from_paths(org_path, rep_path, display_results=True)

    with status_output:
        clear_output(wait=True)
        print(f"Finished. Uploaded files saved under: {upload_dir}")


analyze_button.on_click(on_analyze_pair)

display(widgets.VBox([
    widgets.HTML(
        "<h3 style='margin:0 0 8px 0'>Upload a pair of dog images</h3>"
        "<div style='font-size:14px'>This notebook independently runs the key logic needed to judge replacement feasibility and fitting distortion.</div>"
    ),
    widgets.HBox([org_upload, rep_upload], layout=widgets.Layout(gap="12px")),
    analyze_button,
    status_output,
    result_output,
], layout=widgets.Layout(gap="10px")))
